# 03 — Conception et entraînement du CNN

1. Charger les données prétraitées depuis `data/processed/`
2. Définir l'architecture CNN (Conv2D + MaxPooling + BatchNorm + Dropout + Dense)
3. Compiler (optimiseur Adam, perte categorical_crossentropy, métrique accuracy)
4. Vérifier l'architecture avec `model.summary()`
5. Entraîner avec `model.fit()` — callbacks ModelCheckpoint + EarlyStopping
6. Mesurer la durée d'entraînement avec `time`
7. Visualiser les courbes d'apprentissage (accuracy / loss)
8. Le meilleur modèle est sauvegardé dans `models/final_model.h5`

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from numpy import load
from src import build_cnn, train_model, plot_learning_curves
from src.config import PROCESSED_DATA_DIR, REPORTS_FIGURES_DIR

I0000 00:00:1792400321.142472   40247 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792400321.604648   40247 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792400323.110002   40247 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
x_train = load(PROCESSED_DATA_DIR / "x_train.npy")
x_test = load(PROCESSED_DATA_DIR / "x_test.npy")
y_train = load(PROCESSED_DATA_DIR / "y_train.npy")
y_test = load(PROCESSED_DATA_DIR / "y_test.npy")
print(f"Train: {x_train.shape}, Test: {x_test.shape}")

Train: (5187, 224, 224, 3), Test: (1297, 224, 224, 3)


In [4]:
model = build_cnn()
model.summary()

E0000 00:00:1792400328.840459   40247 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 50176)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    12,845,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,269,060 (50.62 MB)

 Trainable params: 13,268,100 (50.61 MB)

 Non-trainable params: 960 (3.75 KB)

In [5]:
result = train_model(model, x_train, y_train, x_test, y_test)

Epoch 1/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5356 - loss: 3.1409
Epoch 1: val_accuracy improved from None to 0.25058, saving model to /home/brahim-44/Desktop/brainscan-ai/models/best_model.h5



Epoch 1: finished saving model to /home/brahim-44/Desktop/brainscan-ai/models/best_model.h5
163/163 ━━━━━━━━━━━━━━━━━━━━ 334s 2s/step - accuracy: 0.5356 - loss: 3.1409 - val_accuracy: 0.2506 - val_loss: 3.7232
Epoch 2/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5097 - loss: 1.2509
Epoch 2: val_accuracy improved from 0.25058 to 0.36469, saving model to /home/brahim-44/Desktop/brainscan-ai/models/best_model.h5



Epoch 2: finished saving model to /home/brahim-44/Desktop/brainscan-ai/models/best_model.h5
163/163 ━━━━━━━━━━━━━━━━━━━━ 329s 2s/step - accuracy: 0.5097 - loss: 1.2509 - val_accuracy: 0.3647 - val_loss: 2.4100
Epoch 3/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5730 - loss: 1.0486
Epoch 3: val_accuracy improved from 0.36469 to 0.53123, saving model to /home/brahim-44/Desktop/brainscan-ai/models/best_model.h5



Epoch 3: finished saving model to /home/brahim-44/Desktop/brainscan-ai/models/best_model.h5
163/163 ━━━━━━━━━━━━━━━━━━━━ 329s 2s/step - accuracy: 0.5730 - loss: 1.0486 - val_accuracy: 0.5312 - val_loss: 1.2035
Epoch 4/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5726 - loss: 1.0015
Epoch 4: val_accuracy improved from 0.53123 to 0.65305, saving model to /home/brahim-44/Desktop/brainscan-ai/models/best_model.h5



Epoch 4: finished saving model to /home/brahim-44/Desktop/brainscan-ai/models/best_model.h5
163/163 ━━━━━━━━━━━━━━━━━━━━ 353s 2s/step - accuracy: 0.5726 - loss: 1.0015 - val_accuracy: 0.6530 - val_loss: 0.8210
Epoch 5/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5830 - loss: 0.9511
Epoch 5: val_accuracy did not improve from 0.65305
163/163 ━━━━━━━━━━━━━━━━━━━━ 315s 2s/step - accuracy: 0.5830 - loss: 0.9511 - val_accuracy: 0.6446 - val_loss: 0.8905
Epoch 6/25
 47/163 ━━━━━━━━━━━━━━━━━━━━ 4:08 2s/step - accuracy: 0.6263 - loss: 0.9110

KeyboardInterrupt: 

In [ ]:
plot_learning_curves(result["history"], REPORTS_FIGURES_DIR)